# JiraAI — Debug run_agent Step by Step
Inspect every intermediate output in the agentic loop.

In [5]:
import vertexai
import os
import json
from dotenv import load_dotenv
from vertexai.generative_models import (
    GenerativeModel, Tool, FunctionDeclaration,
    Part, Content, GenerationConfig
)

load_dotenv()
vertexai.init(project=os.getenv('PROJECT_ID'), location=os.getenv('LOCATION', 'us-central1'))
print('Vertex AI initialized')

Vertex AI initialized


## 1. Tool Declaration
Apa yang dikirim ke Gemini sebagai definisi tool?

In [6]:
create_ticket_decl = FunctionDeclaration(
    name='create_ticket',
    description='Create a new Jira ticket.',
    parameters={
        'type': 'object',
        'properties': {
            'project_key': {'type': 'string'},
            'title':       {'type': 'string'},
            'description': {'type': 'string'},
            'issue_type':  {'type': 'string', 'enum': ['Bug', 'Story', 'Task']},
            'priority':    {'type': 'string', 'enum': ['Highest', 'High', 'Medium', 'Low', 'Lowest']},
        },
        'required': ['project_key', 'title', 'description', 'issue_type', 'priority'],
    },
)

TOOL = Tool(function_declarations=[create_ticket_decl])

# Inspect the declaration
print(type(TOOL))
print(TOOL)

<class 'vertexai.generative_models._generative_models.Tool'>
function_declarations {
  name: "create_ticket"
  description: "Create a new Jira ticket."
  parameters {
    type_: OBJECT
    properties {
      key: "title"
      value {
        type_: STRING
      }
    }
    properties {
      key: "project_key"
      value {
        type_: STRING
      }
    }
    properties {
      key: "priority"
      value {
        type_: STRING
        enum: "Highest"
        enum: "High"
        enum: "Medium"
        enum: "Low"
        enum: "Lowest"
      }
    }
    properties {
      key: "issue_type"
      value {
        type_: STRING
        enum: "Bug"
        enum: "Story"
        enum: "Task"
      }
    }
    properties {
      key: "description"
      value {
        type_: STRING
      }
    }
    required: "project_key"
    required: "title"
    required: "description"
    required: "issue_type"
    required: "priority"
    property_ordering: "project_key"
    property_ordering: "

## 2. History — Struktur Awal
History sebelum dan sesudah append user message.

In [39]:
history = []
print('history (empty):', history)

user_message = """
Project key: PROJ.
Buat tiket bug: login button tidak berfungsi di mobile.
Priority: High.
Description: Ketika user tap tombol login di iOS Safari, tidak ada response sama sekali. 
Sudah terjadi sejak update v2.3.
"""

history.append(Content(role='user', parts=[Part.from_text(user_message)]))

print('\nhistory setelah append:')
for i, content in enumerate(history):
    print(f'    [{i}] role={content.role}')
    for j, part in enumerate(content.parts):
        print(f'      part[{j}] text={part.text!r}')

history (empty): []

history setelah append:
    [0] role=user
      part[0] text='\nProject key: PROJ.\nBuat tiket bug: login button tidak berfungsi di mobile.\nPriority: High.\nDescription: Ketika user tap tombol login di iOS Safari, tidak ada response sama sekali. \nSudah terjadi sejak update v2.3.\n'


## 3. model.generate_content() — Raw Response
Apa yang Gemini kembalikan?

In [21]:
SYSTEM_PROMPT = 'You are JiraAI, a helpful assistant that manages Jira tickets.'

model = GenerativeModel(
    model_name='gemini-2.0-flash-001',
    system_instruction=SYSTEM_PROMPT,
    tools=[TOOL],
    generation_config=GenerationConfig(temperature=0.1, candidate_count=1),
)

response = model.generate_content(history)

print('type(response)          :', type(response))

/home/ariqlubis/Documents/jiraai/.venv/lib/python3.11/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


type(response)          : <class 'vertexai.generative_models._generative_models.GenerationResponse'>


In [22]:
print(response)

candidates {
  content {
    role: "model"
    parts {
      function_call {
        name: "create_ticket"
        args {
          fields {
            key: "title"
            value {
              string_value: "login button tidak berfungsi di mobile"
            }
          }
          fields {
            key: "project_key"
            value {
              string_value: "PROJ"
            }
          }
          fields {
            key: "priority"
            value {
              string_value: "High"
            }
          }
          fields {
            key: "issue_type"
            value {
              string_value: "Bug"
            }
          }
          fields {
            key: "description"
            value {
              string_value: "Ketika user tap tombol login di iOS Safari, tidak ada response sama sekali. Sudah terjadi sejak update v2.3."
            }
          }
        }
      }
    }
  }
  finish_reason: STOP
  avg_logprobs: -0.013025799523229185
}
usage_

In [23]:
print('len(candidates)         :', len(response.candidates))

len(candidates)         : 1


In [24]:
print('finish_reason           :', response.candidates[0].finish_reason)

finish_reason           : 1


In [25]:
print('usage_metadata          :', response.usage_metadata)

usage_metadata          : prompt_token_count: 110
candidates_token_count: 46
total_token_count: 156
prompt_tokens_details {
  modality: TEXT
  token_count: 110
}
candidates_tokens_details {
  modality: TEXT
  token_count: 46
}



## 4. candidate — Isi Lengkap

In [26]:
candidate = response.candidates[0]

print('type(candidate)         :', type(candidate))
print('candidate.finish_reason :', candidate.finish_reason)

type(candidate)         : <class 'vertexai.generative_models._generative_models.Candidate'>
candidate.finish_reason : 1


In [27]:
print('type(candidate.content) :', type(candidate.content))

type(candidate.content) : <class 'vertexai.generative_models._generative_models.Content'>


In [28]:
print('candidate.content.role  :', candidate.content.role)

candidate.content.role  : model


In [29]:
print('len(content.parts)      :', len(candidate.content.parts))

len(content.parts)      : 1


## 5. Parts — Text vs Function Call
Satu response bisa punya multiple parts.

In [30]:
for i, part in enumerate(candidate.content.parts):
    print(f'--- part[{i}] ---')
    print('  type(part):', type(part))

    # Text part
    try:
        print('  .text     :', repr(part.text))
    except Exception:
        print('  .text     : (not a text part)')

    # Function call part
    try:
        fc = part.function_call
        print('  .function_call.name :', fc.name)
        print('  .function_call.args :', dict(fc.args))
    except Exception:
        print('  .function_call: (not a function call part)')

--- part[0] ---
  type(part): <class 'vertexai.generative_models._generative_models.Part'>
  .text     : (not a text part)
  .function_call.name : create_ticket
  .function_call.args : {'description': 'Ketika user tap tombol login di iOS Safari, tidak ada response sama sekali. Sudah terjadi sejak update v2.3.', 'issue_type': 'Bug', 'priority': 'High', 'project_key': 'PROJ', 'title': 'login button tidak berfungsi di mobile'}


## 6. Ekstrak Function Calls
Cara kita deteksi apakah Gemini minta jalankan tool.

In [31]:
fn_calls = [
    part.function_call
    for part in candidate.content.parts
    if hasattr(part, 'function_call') and part.function_call.name
]

print('Jumlah function calls:', len(fn_calls))
for fn in fn_calls:
    print('  name :', fn.name)
    print('  args :', dict(fn.args))

Jumlah function calls: 1
  name : create_ticket
  args : {'description': 'Ketika user tap tombol login di iOS Safari, tidak ada response sama sekali. Sudah terjadi sejak update v2.3.', 'issue_type': 'Bug', 'priority': 'High', 'project_key': 'PROJ', 'title': 'login button tidak berfungsi di mobile'}


## 7. History setelah model turn di-append

In [32]:
history.append(candidate.content)

print(f'Total turns in history: {len(history)}')
for i, content in enumerate(history):
    print(f'  [{i}] role={content.role}, parts={len(content.parts)}')

Total turns in history: 2
  [0] role=user, parts=1
  [1] role=model, parts=1


## 8. Simulate _dispatch() — Mock (tanpa Jira credentials)

In [34]:
def mock_dispatch(name: str, args: dict) -> str:
    """Simulate tool execution without real Jira connection."""
    if name == 'create_ticket':
        return json.dumps({
            'key': 'PROJ-42',
            'url': 'https://yourorg.atlassian.net/browse/PROJ-42',
            'message': 'Ticket PROJ-42 created successfully'
        })
    return json.dumps({'error': f'Unknown tool: {name}'})


tool_response_parts = []
for fn in fn_calls:
    result = mock_dispatch(fn.name, dict(fn.args))
    print(f'Tool result for {fn.name!r}:')
    print(' ', result)

    tool_response_parts.append(
        Part.from_function_response(
            name=fn.name,
            response={'content': result},
        )
    )

print('\ntype(tool_response_parts[0]):', type(tool_response_parts[0]))

Tool result for 'create_ticket':
  {"key": "PROJ-42", "url": "https://yourorg.atlassian.net/browse/PROJ-42", "message": "Ticket PROJ-42 created successfully"}

type(tool_response_parts[0]): <class 'vertexai.generative_models._generative_models.Part'>


## 9. Feed Tool Result Balik ke Gemini
Append sebagai 'user' role, lalu generate lagi.

In [35]:
history.append(Content(role='user', parts=tool_response_parts))

print(f'History sekarang ({len(history)} turns):')
for i, c in enumerate(history):
    print(f'  [{i}] role={c.role}, parts={len(c.parts)}')

History sekarang (3 turns):
  [0] role=user, parts=1
  [1] role=model, parts=1
  [2] role=user, parts=1


## 10. Second generate_content — Final Response
Seharusnya tidak ada function call lagi, hanya teks.

In [37]:
response2 = model.generate_content(history)
candidate2 = response2.candidates[0]

fn_calls2 = [
    part.function_call
    for part in candidate2.content.parts
    if hasattr(part, "function_call") 
    and part.function_call is not None        # tambah ini
    and part.function_call.name               # baru cek name
]

print('Function calls in second response:', len(fn_calls2))
print('finish_reason                     :', candidate2.finish_reason)

final_text = '\n'.join(
    part.text for part in candidate2.content.parts
    if hasattr(part, 'text') and part.text
)
print('\n=== FINAL RESPONSE ===')
print(final_text)

Function calls in second response: 0
finish_reason                     : 1

=== FINAL RESPONSE ===
OK. Saya sudah membuat tiket dengan key PROJ-42.


## 11. Full History Dump
Semua turn dari awal sampai akhir.

In [38]:
history.append(candidate2.content)

print(f'=== FULL HISTORY ({len(history)} turns) ===\n')
for i, content in enumerate(history):
    print(f'[{i}] role={content.role}')
    for j, part in enumerate(content.parts):
        try:
            if part.text:
                print(f'     part[{j}] TEXT: {part.text[:120]!r}')
        except Exception:
            pass
        try:
            fc = part.function_call
            if fc.name:
                print(f'     part[{j}] FUNCTION_CALL: {fc.name}({dict(fc.args)})')
        except Exception:
            pass
        try:
            fr = part.function_response
            if fr.name:
                print(f'     part[{j}] FUNCTION_RESPONSE: {fr.name} → {str(fr.response)[:80]}')
        except Exception:
            pass
    print()

=== FULL HISTORY (4 turns) ===

[0] role=user
     part[0] TEXT: '\nProject key: PROJ.\nBuat tiket bug: login button tidak berfungsi di mobile.\nPriority: High.\nDescription: Ketika user tap'

[1] role=model
     part[0] FUNCTION_CALL: create_ticket({'description': 'Ketika user tap tombol login di iOS Safari, tidak ada response sama sekali. Sudah terjadi sejak update v2.3.', 'issue_type': 'Bug', 'priority': 'High', 'project_key': 'PROJ', 'title': 'login button tidak berfungsi di mobile'})

[2] role=user
     part[0] FUNCTION_RESPONSE: create_ticket → <proto.marshal.collections.maps.MapComposite object at 0x7f7e29101750>

[3] role=model
     part[0] TEXT: 'OK. Saya sudah membuat tiket dengan key PROJ-42.'

